# Policy Results: Current Par vs. Smart Par vs. Reorder Point
Settings learned on Jan–Jun 2025, tested on Jul–Dec 2025 (35 supplies). Run `python src/run_analysis.py` first to create the results files.

In [1]:
import pandas as pd

summary = pd.read_csv("../reports/summary.csv", index_col="policy")
summary.T.round(3)

policy,Current par,Reorder point,Smart par
total_cost,3584.109,2510.912,4100.055
avg_inventory_value,6777.172,11949.082,10871.087
orders,910.000,335.000,910.000
units_short,1163.000,64.000,231.000
stockout_days,75.000,5.000,13.000
fill_rate,0.991,0.999,0.998
items_with_stockouts,12.000,5.000,7.000


## What the results show

| What it shows | What it means |
|---|---|
| **Current par: 1,163 units short, 12 items ran out** | The 2-week habit fails on some items |
| **Smart par: 231 short, 7 items** | Same Monday routine, smarter numbers: shortages drop 80% |
| **Reorder point: 64 short, 5 items** | Checking daily is safest, with 63% fewer orders (335 vs. 910) |
| **Inventory value: $6,777 → about $11,000** | Being in stock costs money, mostly in expensive CSTD adaptors |

In [4]:
results = pd.read_csv("../reports/results_by_item.csv")
cp = results[results["policy"] == "Current par"]
cp.sort_values("units_short", ascending=False)[["item_name", "category", "units_short", "stockout_days"]].head(10)

,item_name,category,units_short,stockout_days
51,CSTD syringe adaptor,CSTD,469.0,14
54,CSTD bag spike adaptor,CSTD,235.0,10
45,CSTD vial adaptor 20 mm,CSTD,119.0,7
63,Hair cover,PPE,104.0,7
81,Chemo transport bag,Transport,81.0,2
48,CSTD vial adaptor 13 mm,CSTD,68.0,12
75,Non-PVC IV admin set,Admin sets,26.0,3
72,0.2 micron in-line filter,Admin sets,25.0,9
87,Sharps container (chemo),Waste,17.0,7
102,Amber line covers,Light protection,9.0,2


## Key takeaway
CSTD adaptors account for 891 of the 1,163 units short under the current par (77%). They have a 7-day manufacturer lead time, so a 2-week par can't cover a one-week order cycle plus a one-week delivery. One par rule for every item over-stocks next-day items and under-stocks long-lead items.

## Validation
- **Formulas cross-checked in Excel:** safety stock, reorder point, order quantity and smart par match an independent Excel calculation for 5 sample items, including a hand calculation for the CSTD syringe adaptor (safety stock 189.4, reorder point 417.4).
- **Simulation tested against a hand trace:** a 5-day paper example (par 20, usage 6, 8, 5, 7, 9) gives 9 units short and a 74.3% fill rate, by hand and in code. This is an automated test in `tests/test_simulate.py`.
- **Common-sense checks pass:** fill rates are between 0 and 1, the current par places 910 orders (35 items × 26 Mondays), and inventory is never negative.
- **Method check:** measuring variability on raw daily data (weekends included) instead of weekly totals adds about $500 of inventory, so small method choices change real dollars.